# RetailOps 360 — Fulfillment & Delivery Performance

Queries and saved result previews for reading on GitHub. Previews may not contain the full query results. See the [original notebook exports](original_exports/) for the supplied files and [all questions and SQL answers](../BUSINESS_QUESTIONS_AND_SQL.md) for quick access.


# Segment 3 - Fulfillment & Delivery Performance Analysis

## Analysis 07 — Completed Orders Without Shipments

**Business question:** Which completed orders have no shipment record, indicating a potential fulfillment backlog?


In [ ]:
%sql
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.expected_delivery_date,
    o.order_amount
FROM retailops360.source.orders AS o
LEFT ANTI JOIN retailops360.source.shipments AS s
    ON o.order_id = s.order_id
WHERE o.order_status = 'Completed'
ORDER BY o.order_date, o.order_id;

order_id,customer_id,order_date,expected_delivery_date,order_amount
1150,2913,2024-01-01,2024-01-05,1186.00
13243,5053,2024-01-01,2024-01-08,787.00
4808,6394,2024-01-02,2024-01-11,1682.00
44502,9515,2024-01-02,2024-01-11,463.00
52719,1588,2024-01-02,2024-01-10,884.00
53830,7782,2024-01-02,2024-01-10,71.00
21407,7432,2024-01-03,2024-01-12,986.00
24718,11503,2024-01-03,2024-01-09,3111.00
30148,3867,2024-01-03,2024-01-11,2201.00
33609,1162,2024-01-03,2024-01-07,4502.00


## Analysis 08 — Late Delivery Performance

**Business question:** Which delivered orders arrived after their expected delivery date, and how many days late were they?


In [ ]:
%sql
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.expected_delivery_date,
    s.delivered_date,
    s.shipping_method,
    DATEDIFF(
        s.delivered_date,
        o.expected_delivery_date
    ) AS days_late
FROM retailops360.source.orders AS o
INNER JOIN retailops360.source.shipments AS s
    ON o.order_id = s.order_id
WHERE s.shipment_status = 'Delivered'
    AND s.delivered_date > o.expected_delivery_date
ORDER BY days_late DESC, o.order_id;

order_id,customer_id,order_date,expected_delivery_date,delivered_date,shipping_method,days_late
20,7892,2025-03-11,2025-03-14,2025-03-19,Standard,5
27,1767,2026-01-29,2026-02-05,2026-02-10,Standard,5
31,12304,2026-09-28,2026-10-04,2026-10-09,Express,5
43,9439,2026-04-06,2026-04-09,2026-04-14,Standard,5
60,9639,2026-07-22,2026-07-31,2026-08-05,Standard,5
69,8967,2026-09-17,2026-09-24,2026-09-29,Standard,5
81,1793,2024-10-27,2024-10-30,2024-11-04,Express,5
88,1032,2025-10-03,2025-10-10,2025-10-15,Express,5
103,9541,2024-08-04,2024-08-10,2024-08-15,Standard,5
130,1952,2024-10-10,2024-10-14,2024-10-19,Standard,5


## Analysis 09 — Fully Paid Orders Without Confirmed Delivery

**Business question:** Which fully paid, completed orders have no shipment marked as delivered, including orders with other shipment statuses?


In [ ]:
%sql
WITH captured_payments AS (
    SELECT
        order_id,
        ROUND(SUM(payment_amount), 2)
            AS total_paid
    FROM retailops360.source.payments
    WHERE payment_status = 'Captured'
    GROUP BY order_id
),

fully_paid_orders AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_date,
        o.order_amount,
        p.total_paid
    FROM retailops360.source.orders AS o
    INNER JOIN captured_payments AS p
        ON o.order_id = p.order_id
    WHERE o.order_status = 'Completed'
      AND ABS(o.order_amount - p.total_paid) < 0.01
),

undelivered_orders AS (
    SELECT
        f.*
    FROM fully_paid_orders AS f
    LEFT ANTI JOIN retailops360.source.shipments AS s
        ON f.order_id = s.order_id
       AND s.shipment_status = 'Delivered'
)

SELECT
    u.order_id,
    u.customer_id,
    u.order_date,
    u.order_amount,
    u.total_paid,
    COALESCE(
        s.shipment_status,
        'No Shipment Record'
    ) AS shipping_status
FROM undelivered_orders AS u
LEFT JOIN retailops360.source.shipments AS s
    ON u.order_id = s.order_id
ORDER BY u.order_date, u.order_id;

order_id,customer_id,order_date,order_amount,total_paid,shipping_status
1150,2913,2024-01-01,1186.00,1186.00,No Shipment Record
13243,5053,2024-01-01,787.00,787.00,No Shipment Record
4808,6394,2024-01-02,1682.00,1682.00,No Shipment Record
23350,5302,2024-01-02,517.00,517.00,Lost
34721,9233,2024-01-02,1334.00,1334.00,Returned to Sender
44502,9515,2024-01-02,463.00,463.00,No Shipment Record
51265,2701,2024-01-02,1371.00,1371.00,Returned to Sender
53830,7782,2024-01-02,71.00,71.00,No Shipment Record
15497,7171,2024-01-03,1484.00,1484.00,Lost
21407,7432,2024-01-03,986.00,986.00,No Shipment Record
